In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from mlforecast import MLForecast
from tinyconformal.series import ConformalizedQuantileTimeSeriesRegressor
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression


In [2]:
# Generate synthetic panel data for five time series
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

# LightGBM

In [3]:

#def create_mlforecast_multiquantile():
#        import lightgbm as lgb
#
#     # Define 90% and 50% prediction interval quantile LightGBM models
#     models = {
#         "LGBM-lo-90": lgb.LGBMRegressor(objective="quantile", alpha=0.05, random_state=42),
#         "LGBM-hi-90": lgb.LGBMRegressor(objective="quantile", alpha=0.95, random_state=42),
#         "LGBM-lo-50": lgb.LGBMRegressor(objective="quantile", alpha=0.25, random_state=42),
#         "LGBM-hi-50": lgb.LGBMRegressor(objective="quantile", alpha=0.75, random_state=42),
#     }
#     return MLForecast(
#         models=models,
#         freq="MS",
#         lags=[1, 7],
#     )
#models = create_mlforecast_multiquantile()

# RandomForestQuantileRegressor

In [4]:

class QuantileRF(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        quantile=0.5,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
        **kwargs,
    ):
        self.quantile = quantile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.kwargs = kwargs

    def fit(self, X, y):
        self.model_ = RandomForestQuantileRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            **self.kwargs,
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X, quantiles=self.quantile)

In [5]:
def model_callable():
    models = {
        "RF-lo-90": QuantileRF(
            quantile=0.05, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-hi-90": QuantileRF(
            quantile=0.95, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-lo-50": QuantileRF(
            quantile=0.25, n_estimators=100, max_depth=8, random_state=42
        ),
        "RF-hi-50": QuantileRF(
            quantile=0.75, n_estimators=100, max_depth=8, random_state=42
        ),
        "RF-50": QuantileRF(
            quantile=0.50, n_estimators=100, max_depth=8, random_state=42
        ),
        #"LinearRegression": LinearRegression()
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )


models = model_callable()

In [6]:
cqr = ConformalizedQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     intervals=[("RF-lo-90", "RF-hi-90"), ("RF-lo-50", "RF-hi-50")],
     n_windows=10,
     nexcp=True,
     weighted_refit=False,
 )
cqr.fit(train, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,intervals,"[('RF-lo-90', ...), ('RF-lo-50', ...)]"
,n_windows,10
,nexcp,True
,decay,0.99
,weighted_refit,False
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [7]:
cqr.predict_interval(h=12)

,unique_id,ds,RF-lo-90,RF-hi-90,RF-lo-50,RF-hi-50,RF-50,RF-lo-90-cqr,RF-hi-90-cqr,RF-lo-50-cqr,RF-hi-50-cqr
0,1,1960-01-01,337.00,436.5,363.0,406.00,396.0,331.30,442.20,360.00,409.00
1,1,1960-02-01,305.00,548.0,353.0,406.00,405.0,296.00,557.00,354.00,405.00
2,1,1960-03-01,301.00,559.0,360.0,406.00,396.0,259.00,601.00,327.00,439.00
3,1,1960-04-01,301.00,559.0,342.0,405.00,405.0,260.00,600.00,320.00,427.00
4,1,1960-05-01,300.15,559.0,337.0,405.00,379.5,264.15,595.00,301.00,441.00
5,1,1960-06-01,277.00,559.0,336.0,405.00,361.0,250.85,585.15,299.00,442.00
6,1,1960-07-01,276.10,559.0,318.0,398.25,363.0,254.10,581.00,261.00,455.25
7,1,1960-08-01,277.00,559.0,318.0,405.00,363.0,219.00,617.00,260.00,463.00
8,1,1960-09-01,233.00,559.0,318.0,398.25,363.0,214.00,578.00,272.00,444.25
9,1,1960-10-01,211.00,559.0,331.5,405.00,363.0,219.00,551.00,309.75,426.75


In [8]:
cqr.evaluate(test, h=12)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,RF,50%,0.5,0.167,69.188,367.354
1,RF,90%,0.1,0.833,282.354,465.688
2,RF-cqr,50%,0.5,0.250,123.646,316.896
3,RF-cqr,90%,0.1,0.917,321.829,390.163


# GradientBoostingRegressor & HistGradientBoostingRegressor

In [9]:
def model_callable():
    models = {
        "GBR-lo-90": GradientBoostingRegressor(loss="quantile", alpha=0.05, n_estimators=100, random_state=42),
        "GBR-hi-90": GradientBoostingRegressor(loss="quantile", alpha=0.95, n_estimators=100, random_state=42),
        "HGB-lo-90": HistGradientBoostingRegressor(loss="quantile", quantile=0.05, max_iter=100, random_state=42),
        "HGB-hi-90": HistGradientBoostingRegressor(loss="quantile", quantile=0.95, max_iter=100, random_state=42),
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )

models = model_callable()

In [10]:
cqr = ConformalizedQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     intervals=[
         ("GBR-lo-90", "GBR-hi-90"),
         ("HGB-lo-90", "HGB-hi-90"),
     ],
     n_windows=10,
     nexcp=True,
     weighted_refit=True,
 )
cqr.fit(train, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,intervals,"[('GBR-lo-90', ...), ('HGB-lo-90', ...)]"
,n_windows,10
,nexcp,True
,decay,0.99
,weighted_refit,True
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [11]:
cqr.predict_interval(h=12)

,unique_id,ds,GBR-lo-90,GBR-hi-90,HGB-lo-90,HGB-hi-90,GBR-lo-90-cqr,GBR-hi-90-cqr,HGB-lo-90-cqr,HGB-hi-90-cqr
0,1,1960-01-01,341.632739,432.116066,339.899488,558.914626,373.632735,400.116070,350.899488,547.914626
1,1,1960-02-01,304.961630,493.339989,303.170911,558.914626,308.962028,489.339590,322.170911,539.914626
2,1,1960-03-01,270.954151,546.504506,270.653111,558.914626,224.702347,592.756311,262.653111,566.914626
3,1,1960-04-01,241.326868,519.685234,244.496236,558.914626,296.330142,464.681960,251.495918,551.914944
4,1,1960-05-01,230.633721,508.583260,236.752135,558.914626,275.383813,463.833168,234.752135,560.914626
5,1,1960-06-01,230.665255,508.583260,218.495189,558.914626,229.665255,509.583260,199.495189,577.914626
6,1,1960-07-01,215.022314,508.583260,211.109918,558.914626,182.555358,541.050216,189.109918,580.914626
7,1,1960-08-01,207.015413,508.583260,188.150527,558.914626,173.015732,542.582941,133.615261,613.449892
8,1,1960-09-01,183.967445,508.583260,188.254560,558.914626,235.066011,457.484694,174.254560,572.914626
9,1,1960-10-01,169.256486,508.181546,189.184806,558.914626,162.256885,515.181148,197.184806,550.914626


In [12]:
cqr.evaluate(test, h=12)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,GBR,90%,0.1,0.750,278.748,674.166
1,GBR-cqr,90%,0.1,0.417,245.737,695.204
2,HGB,90%,0.1,0.833,329.577,513.195
3,HGB-cqr,90%,0.1,0.917,339.779,408.255
